In [ ]:
from nicegui import ui
import       # API du groupe 2
import time
import threading


class PowerMeterGUI:
    def __init__(self):
        self.connected = False
        self.measuring = False

        ui.label("Interface Puissancemètre").classes("text-2xl mt-4 mb-4")

        # --- Affichage puissance ---
        self.power_label = ui.label("-- W").classes("text-4xl my-4")

        # --- Statut ---
        self.status_label = ui.label("Statut : Déconnecté").classes("text-gray-500 mb-4")

        # --- Boutons ---
        with ui.row().classes("gap-4"):
            ui.button("Connecter", on_click=self.connect).classes("bg-green-600 text-white")
            ui.button("Déconnecter", on_click=self.disconnect).classes("bg-red-600 text-white")

        ui.button("Mesure instantanée", on_click=self.measure_once).classes("bg-blue-600 text-white my-4")

        with ui.row().classes("gap-4"):
            ui.button("Start continu", on_click=self.start_continuous).classes("bg-purple-600 text-white")
            ui.button("Stop continu", on_click=self.stop_continuous).classes("bg-orange-600 text-white")

    # --- Connexion avec API du groupe 2 ---
    def connect(self):
        try:
            success = powermeter.connect()
            if success:
                self.connected = True
                self.status_label.set_text("Statut : Connecté ✔️")
            else:
                self.status_label.set_text("Erreur : impossible de se connecter")
        except Exception as e:
            self.status_label.set_text(f"Erreur : {e}")

    def disconnect(self):
        try:
            powermeter.disconnect()
            self.connected = False
            self.measuring = False
            self.status_label.set_text("Statut : Déconnecté")
            self.power_label.set_text("-- W")
        except Exception as e:
            self.status_label.set_text(f"Erreur : {e}")

    # --- Mesure instantanée ---
    def measure_once(self):
        if not self.connected:
            self.status_label.set_text("⚠️ Non connecté")
            return

        try:
            value = powermeter.get_power()
            self.power_label.set_text(f"{value:.3f} W")
        except Exception as e:
            self.status_label.set_text(f"Erreur : {e}")

    # --- Mesure continue ---
    def start_continuous(self):
        if not self.connected:
            self.status_label.set_text("⚠️ Non connecté")
            return
        if self.measuring:
            return

        self.measuring = True
        threading.Thread(target=self.continuous_measure_loop, daemon=True).start()

    def stop_continuous(self):
        self.measuring = False

    def continuous_measure_loop(self):
        while self.measuring:
            try:
                value = powermeter.get_power()
                self.power_label.set_text(f"{value:.3f} W")
            except Exception as e:
                self.status_label.set_text(f"Erreur : {e}")
                self.measuring = False
            time.sleep(0.5)


# --- Lancement de l’application NiceGUI ---
PowerMeterGUI()
ui.run()0


ModuleNotFoundError: No module named 'powermeter'